<div style="width:100%; background-color:#181818; color:#f1f1f1; padding:30px 0; text-align:center; border-radius:10px;">

  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExcXVodWNsM3Bia3duZGljZzRqMTI2MGFiZjlkZzBwcmhuaWxydjlpaiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/AOSwwqVjNZlDO/giphy.gif" alt="Beam Oscillation" width="400" style="border-radius:10px;">

  <h3 style="color:#ffffff; margin-top:15px;"><b>Manual Flexural Method. all deformations</b></h3>

  <p><b>Author:</b> <a href="http://caceli.net/" style="color:#3ea6ff; text-decoration:none;">Msc. Ing. Carlos Andrés Celi Sánchez</a></p>
  <p><b>Course:</b> Matrix Analisys</p>
  <p><b>Year:</b> APRIL - 2026</p>

</div>

#### Libraries

In [1]:
import numpy as np
import pandas as pd

![alt text](image-3.png)

#### Data

In [2]:
w = 3                       # T/m
L = 5                       # m
vb = 0.30                   # m
vh = 0.50                   # m
E = 2000000                 # T/m2
T1 = 40                     # celcius
T2 = 15                     # celcius
alfa = 1.3e-5               # m/m/celcius 
dP = 0.0001                 # previus deformation
dR1 = 0.002                 # rad
dR2 = -0.012                # m
dR3 = -0.025                # m

#### Calculations

In [3]:
I = vb*vh**3 / 12

##### Flex Matrix
![alt text](image-4.png)

In [4]:
F11 = L / (3*E*I)
F21 = -L / (6*E*I)
F31 = 0
F12 = F21
F22 = 2*L / (3*E*I)
F32 = L / (6*E*I)
F13 = F31
F23 = F32
F33 = 2*L / (3*E*I)

F = np.array([[F11, F12, F13],
              [F21, F22, F23],
              [F31, F32, F33]])

F_df = pd.DataFrame(F, columns=['Q1','Q2','Q3'], index = ['Q1','Q2','Q3'])
F_df

,Q1,Q2,Q3
Q1,0.000267,-0.000133,0.000000
Q2,-0.000133,0.000533,0.000133
Q3,0.000000,0.000133,0.000533


##### Displacement vector corresponding to the redundants in the real structure $\{D_Q\}$
![alt text](image-5.png)

In [5]:
DQ = np.array([[-dR1],
               [0],
               [0]])

DQ_df = pd.DataFrame(DQ, columns=['DQ'], index = ['Q1','Q2','Q3'])
DQ_df

,DQ
Q1,-0.002
Q2,0.000
Q3,0.000


##### Displacement vector corresponding to the redundants due to real loads in the released or statically determinate structure
$$
\{D_{QL}\}
=
\begin{bmatrix}
D_{QL1} \\
D_{QL2} \\
\vdots \\
D_{QLn}
\end{bmatrix}
$$

In [6]:
DQL = np.array([[-w*L**3 / (24*E*I)],
                [w*L**3 / (24*E*I)],
                [w*L**3 / (16*E*I)]])

DQL_df = pd.DataFrame(DQL, columns=['DQL'], index = ['Q1','Q2','Q3'])
DQL_df

,DQL
Q1,-0.00250
Q2,0.00250
Q3,0.00375


##### Displacement vector corresponding to the redundants due to temperature effects in the released or statically determinate structure

$$
\{D_{QT}\}
=
\begin{bmatrix}
D_{QT1} \\
D_{QT2} \\
\vdots \\
D_{QTn}
\end{bmatrix}
$$

In [7]:
DQT = L /2 * (alfa*(T1-T2)) / vh * np.array([[-1],
                                             [2],
                                             [2]])

DQT_df = pd.DataFrame(DQT, columns=['DQT'], index = ['Q1','Q2','Q3'])
DQT_df

,DQT
Q1,-0.001625
Q2,0.003250
Q3,0.003250


##### Displacement vector corresponding to the redundants due to initial deformations in the released or statically determinate structure

$$
\{D_{QP}\}
=
\begin{bmatrix}
D_{QP1} \\
D_{QP2} \\
\vdots \\
D_{QPn}
\end{bmatrix}
$$

In [8]:
DQP = np.array([[0],
                [dP/L],
                [-2*dP/L]])

DQP_df = pd.DataFrame(DQP, columns=['DQP'], index = ['Q1','Q2','Q3'])
DQP_df

,DQP
Q1,0.00000
Q2,0.00002
Q3,-0.00004


##### Displacement vector corresponding to the redundants due to support displacements in the released or statically determinate structure

$$
\{D_{QR}\}
=
\begin{bmatrix}
D_{QR1} \\
D_{QR2} \\
\vdots \\
D_{QRn}
\end{bmatrix}
$$

In [9]:
DQR = np.array([[-dR2/L],
                [-2*dR2/L + dR3/L],
                [dR2/L - 2*dR3/L]])

DQR_df = pd.DataFrame(DQR, columns=['DQR'], index = ['Q1','Q2','Q3'])
DQR_df

,DQR
Q1,0.0024
Q2,-0.0002
Q3,0.0076


#### Total displacement vector corresponding to the redundants in the released or statically determinate structure

$$
\{D_{QS}\}
=
\{D_{QL}\}
+
\{D_{QT}\}
+
\{D_{QP}\}
+
\{D_{QR}\}
$$

In [10]:
DQS = DQL + DQT + DQP + DQR

DQS_df = pd.DataFrame(DQS, columns=['DQS'], index = ['Q1','Q2','Q3'])
DQS_df

,DQS
Q1,-0.001725
Q2,0.005570
Q3,0.014560


#### Calculation of redundants

$$
\{Q\}
=
[F]^{-1}
\left(
\{D_Q\}
-
\{D_{QS}\}
\right)
$$

In [11]:
#------------Due all deformations---------------
Q_all = np.linalg.inv(F) @ (DQ - DQS)
Q_all_df = pd.DataFrame(Q_all, columns=['Q due all deformations'])
#------------only due real loads---------------
Q_L = np.linalg.inv(F) @ (0 - DQL)
Q_L_df = pd.DataFrame(Q_L, columns=['Q due real loads'])
#------------only due climate effects---------------
Q_T = np.linalg.inv(F) @ (0 - DQT)
Q_T_df = pd.DataFrame(Q_T, columns=['Q due climate effects'])
#------------only due previus defformations---------------
Q_P = np.linalg.inv(F) @ (0 - DQP)
Q_P_df = pd.DataFrame(Q_P, columns=['Q due previus defformations'])
#------------only due support diplacement---------------
Q_R = np.linalg.inv(F) @ (0 - DQR)
Q_R_df = pd.DataFrame(Q_R, columns=['Q due support diplacement'])

Q = pd.concat([Q_all_df, Q_L_df, Q_T_df, Q_P_df, Q_R_df], axis = 1, ignore_index= False)
Q

,Q due all deformations,Q due real loads,Q due climate effects,Q due previus defformations,Q due support diplacement
0,-3.416827,9.014423,4.21875,-0.034615,-7.961538
1,-4.771154,-0.721154,-3.75000,-0.069231,2.076923
2,-26.107212,-6.850962,-5.15625,0.092308,-14.769231
